# J'ai deux passions : geographie et manifestations

## Introduction

Les manifestations du 17 octobre ont été d'une ampleur rare en France. L'anticipation de cet évènement était telle que certain&middot;es ont crié à la "fin de l'époque contemporaine". On sait depuis Siegfried (1913) que d'un territoire à l'autre, l'expression politique change du tout au tout, et que les facteurs géographiques sont déterminants dans la politique française.

Une très grande hypothèse de ce projet est la suivante : les manifestant&middot;es, dans leur grande bonté, ont mis à disposition de notre groupe une cartographie exhaustive des évènements du 17 octobre. Tout évènement déclaré a bien eu lieu ; en retour, tout évènement qui a eu lieu a bien été déclaré. La vérification empirique de cette hypothèse demanderait sûrement un scan exhaustif de la presse régionale - si tant est que celle-ci ait rencensé toutes les manifestations — qui dépasse un peu le cadre de ce travail. On assimilera donc ici déclaration d'évènement à évènement. 

Ce mouvement, qui revêtait au départ un aspect très flou, s'est doté de structures d'organisation pour permettre une mobilisation coordonnée (entre les participant&middot;es, et pas par un organe central). Cela fait peut-être suite au mouvement des Gilets Jaunes : la carte que nous avons étudiée permettait d'émettre un signal crédible de la présence de militant&middot;es.

Le but de ce projet est double. 
- D'une part, il s'agit d'obtenir un aperçu, quoique partiel, des déterminants géographiques des évènements du 17 octobre. Quelles sont les prédicteurs, au niveau local, de la présence d'un évènement ? 
- De l'autre, il s'agit de se demander l'échelle la plus pertinente pour analyser ces phénomènes, dans la continuité des interrogations en géographie économique sur le *Modifiable Aeral Unit Problem* (MAUP). Et si l'échelle retenue affectait grandement la performance des prédicteurs trouvés auparavant ? 

Importation des modules:

In [1]:
%reload_ext autoreload
%autoreload 2
import pandas as pd
import numpy as np
import folium
import useful.useful_db as udb
import ipywidgets

In [2]:
#pour vérifier quelles fonction on a :
import importlib
importlib.reload(udb)
print([name for name in dir(udb) if not name.startswith("__")])

['MarkerCluster', 'URL_COMMUNES', 'ajouter_code_insee', 'carte_events', 'carti_download', 'couleur_evenement', 'couleurs_actions', 'desc_bool', 'desc_types_events', 'folium', 'gpd', 'ipywidgets', 'merge_communes', 'np', 'pd']


Importation des bases :

In [3]:
# table qu'on a scrapée
scraped = pd.read_csv(
    "/home/onyxia/python_manifestation_explosion/bdd/just17.csv",
    sep=";",
)

# table d'appartenance des communes à des zones géographiques
all_communes = pd.read_excel(
    "/home/onyxia/python_manifestation_explosion/bdd/table-appartenance-geo-communes-2026.xlsx",
    header=5,
    dtype=str,
    engine="calamine",
)



## Aperçu des données

### Valeurs manquantes

Nous avons scrapé ce [site](https://www.17octobregiletjaune.net/carte/17-octobre/), créé par Benjamin Ball, qui permettait à chacun&middot;e de renseigner un point de manifestation pour le 17 octobre, et de se déclarer comme participant&middot;e aux dites manifestations. Ces manifestations sont renseignées dans le fichier `nom de fichier.csv`. Un aperçu rapide de la table donne :

In [ ]:
scraped.head(5)
print(f"Cette table contient {len(scraped)} lignes, correspondant à autant d'évènements.")
print(scraped.columns)


Cette table contient 2237 lignes, correspondant à autant d'évènements.
['id', 'title', 'time', 'date', 'amount', 'ville', 'gazole', 'cp', 'maj', 'nom', 'exploitant', 'adresse', 'lat', 'dep', 'e10', 'annonce', 'heure', 'commune', 'rondpoint', 'lon', 'genre']


On enlève les sites pour lesquels on n'a pas de coordonnées, qui ne pourront de toute façon pas être utilisées

In [5]:
scraped_cleaned = scraped.dropna(subset=["lat","lon"])
print(f"Il y a {len(scraped_cleaned)} lignes où sont renseignées des coordonnées géographiques.")

scraped_dirty = scraped[scraped["lat"].isna()|scraped["lon"].isna()]
print(f"Il y a {len(scraped_dirty)} lignes où ne sont pas renseignées de coordonnées géographiques.")



Il y a 2009 lignes où sont renseignées des coordonnées géographiques.
Il y a 228 lignes où ne sont pas renseignées de coordonnées géographiques.


Penchons nous de plus près sur ces lignes sans coordonnées :

In [6]:
nb_villes = scraped_dirty["ville"].notna().sum()
print(f"Il y a {nb_villes} lignes où la ville est renseignée.")

Il y a 0 lignes où la ville est renseignée.


Elles nous inquiètent moins désormais : on peut les enlever en toute sécurité.

In [7]:
scraped = scraped_cleaned

### Statistiques et cartes descriptives

Sur le site, les participant&middot;es étaient invité&middot;es à renseigner le type d'évènement qui allait se dérouler. Ces types sont renseigéns dans la colonne `title` :

In [8]:
types_events = scraped["title"].unique().tolist()
types_events_freq = scraped["title"].value_counts(dropna=False)
print(types_events_freq)

title
manifestation         606
rassemblement         434
blocage               291
assemblee             206
rond_point            199
barrage_filtrant      131
operation_escargot     46
autre                  40
pancartes              24
tractage               23
prise_de_parole         6
piquet_de_greve         3
Name: count, dtype: int64


NB: une "opération escargot" correspond à un ralentissement volontaire de la circulation par des automobilistes coordonné&middot;es.

En termes de distribution dans l'espace, voici ce qui apparait :

In [9]:
carte_all_events = udb.carte_events(scraped)
carte_all_events

NB : nous avons choisi de garder les DROM COM dans la plupart de nos cartes, dans un souci d'inclusivité.

Afin de faire des statistiques sur la distribution des évènements en France, on y ajoute le code Insee pour chacune des communes. Pour cela, on essaie de croiser directement avec un fond de carte fourni par etalab à [cette adresse](https://etalab-datasets.geo.data.gouv.fr/contours-administratifs/) :

In [ ]:
scraped_insee = udb.ajouter_code_insee(scraped)
print(f"On n'a pas de perte : il y a {len(scraped_insee) - len(scraped)} lignes différentes entre `scraped` et `scraped_insee"

On n'a pas de perte : il y a 0 lignes différentes entre `scraped` et `scraped_insee`


On la croise ensuite avec une table avec toutes les communes françaises, fournie par l'Insee.

In [11]:
all_communes_events = udb.merge_communes(scraped_insee, all_communes)
len(all_communes_events)

35737

Cela nous permet de faire des statistiques à l'échelle de la France. 

In [12]:
udb.desc_types_events(all_communes_events)

,Effectif,Proportion (%)
Par types d'évènements,,
NaN,33751,94.4
Manifestation,597,1.7
Rassemblement,425,1.2
Blocage,289,0.8
Assemblee,206,0.6
Rond point,198,0.6
Barrage filtrant,129,0.4
Operation escargot,46,0.1
Autre,40,0.1


In [13]:
udb.desc_bool(all_communes_events)

,Effectif,Proportion (%)
has_event,,
Sans événement,33751,96.78
Avec événement,1124,3.22


## Impact des variables socio-démographiques

## Quelle échelle ?

Le MAUP est un problème qui se pose en premier lieu à celui ou celle qui tente de donner des explications aux phénomènes géographiques. Il est prouvé depuis que l'échelle de l'analyse change les résultats, et ce de deux manières (Briant et al., 2010) : 
- l'agglomération des zones, c'est-à-dire leur regroupement en des ensembles plus ou moins grans, change la donne (de la commune, au département, à la région)
- la forme des zones, c'est-à-dire le découpage de ces zones (commune VS zone d'emploi)

Schématiquement, cela donne : 

![Fig 1 : Briant et al., 2010](modele_echelle/MAUP_simple.png)

Nous allons ici essayer de comparer les performances des modèles prédictifs, en fonction des découpages géographiques choisis.

In [14]:
import modele_echelle.preprocessing as pp
import statsmodels.formula.api as smf

IndentationError: expected an indented block after function definition on line 9 (preprocessing.py, line 11)

### Par taille de la zone

Intéressons-nous d'abord à la taille de la zone retenue. On peut ici faire des `group_by` à différentes échelles, pour voir les effets des agglomérations. On utilise ici la régression logistique, qui modélise les log-odds $\ln (\frac{p}{1-p})$ comme une fonction linéaire des variables explicatives, dont les paramètres sont estimés avec le maximum de vraisemblance (MLE). On essaie d'expliquer une variable binaire $Y$, correspondant à la colonne `has_event` de notre table `all_communes_event`. On a donc $p=P(Y=1 | X)$.

Formellement, on a donc : 

$$\ln\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1 x_1 + \dots + \beta_k x_k$$

Ainsi, par département, on obtient :

### Par forme de la zone